# S2 K-Scan 實驗 - Google Colab 版本

此 Notebook 用於在 Google Colab 上執行 S2 K-scan 實驗

**實驗目的**: 找出最佳感測器數量 K

**GPU 需求**: Tesla T4 或更高（建議使用 Colab Pro）

**預估時間**: 
- K=30 + K=50 快速測試: 4-6 小時
- 完整 K-scan (30,50,80,100): 16-24 小時
- 包含 K=200: 額外 10-20 小時（高風險）

## 1. 環境設置

### 1.1 檢查 GPU

In [ ]:
!nvidia-smi

### 1.2 掛載 Google Drive（用於保存結果）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 創建備份目錄
!mkdir -p /content/drive/MyDrive/pinns_checkpoints
!mkdir -p /content/drive/MyDrive/pinns_results

### 1.3 Clone 專案（如果尚未 clone）

In [ ]:
import os

if not os.path.exists('/content/pinns-sparse-flow'):
    !git clone https://github.com/YOUR_USERNAME/pinns-sparse-flow.git /content/pinns-sparse-flow
    print("✓ 專案已 clone")
else:
    print("✓ 專案已存在")
    # 更新到最新版本
    %cd /content/pinns-sparse-flow
    !git pull

%cd /content/pinns-sparse-flow

### 1.4 安裝依賴

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q -e .
!pip install -q pyyaml h5py wandb matplotlib seaborn

print("✓ 依賴安裝完成")

### 1.5 下載數據（如果需要）

In [ ]:
# 檢查數據是否存在
import os

dns_data = "/content/pinns-sparse-flow/data/kolmogorov_dns/dns_re50_t100.h5"
sensor_dir = "/content/pinns-sparse-flow/data/sensors/kolmogorov/"

if os.path.exists(dns_data):
    print("✓ DNS 數據已存在")
else:
    print("❌ DNS 數據不存在，請先上傳數據文件")
    print(f"   預期位置: {dns_data}")

# 檢查 sensor 文件
k_values = [30, 50, 80, 100, 200]
for k in k_values:
    sensor_file = f"{sensor_dir}sensors_temporal_K{k}_re50_256x256_t15-35.json"
    if os.path.exists(sensor_file):
        print(f"✓ K={k} sensor 已存在")
    else:
        print(f"❌ K={k} sensor 不存在: {sensor_file}")

## 2. 執行實驗

### 2.1 設定實驗參數

In [ ]:
# 選擇要執行的 K 值
# 選項 1: 快速測試（推薦新手）
K_VALUES = [30, 50]

# 選項 2: 完整測試（論文用）
# K_VALUES = [30, 50, 80, 100]

# 選項 3: 包含高風險 K=200
# K_VALUES = [30, 50, 80, 100, 200]

# 訓練 epochs（Colab 建議使用較少 epochs 快速測試）
EPOCHS = 5000  # 預設 10000，Colab 可用 5000 快速測試

# 是否使用 wandb 記錄（需要登入）
USE_WANDB = False  # 設為 True 則需先執行 !wandb login

print(f"將執行 K 值: {K_VALUES}")
print(f"每個實驗訓練 {EPOCHS} epochs")
print(f"預估總時間: {len(K_VALUES) * (EPOCHS/1000) * 1.5:.1f} 小時")

### 2.2 配置 W&B（可選）

In [ ]:
if USE_WANDB:
    !wandb login
else:
    # 禁用 wandb
    import os
    os.environ['WANDB_MODE'] = 'disabled'
    print("✓ W&B 已禁用")

### 2.3 執行訓練循環

In [ ]:
import subprocess
import time
from datetime import datetime

# 記錄結果
results = {}
start_time = time.time()

for k in K_VALUES:
    print("=" * 60)
    print(f"▶ 開始訓練 K={k}")
    print("=" * 60)
    
    config_file = f"configs/experiments/S2_k_scan/s2_qr_K{k}_2d_re50.yml"
    exp_start = time.time()
    
    try:
        # 執行訓練
        cmd = [
            "python", "scripts/train/train.py",
            "--cfg", config_file,
            "--epochs", str(EPOCHS)
        ]
        
        result = subprocess.run(cmd, capture_output=False, text=True)
        
        exp_duration = time.time() - exp_start
        
        if result.returncode == 0:
            print(f"\n✅ K={k} 訓練成功")
            print(f"耗時: {exp_duration/3600:.2f} 小時")
            results[k] = {'status': 'success', 'time': exp_duration}
            
            # 備份到 Google Drive
            backup_dir = f"/content/drive/MyDrive/pinns_checkpoints/S2_K{k}"
            !mkdir -p {backup_dir}
            !cp -r checkpoints/experiments/S2_qr_K{k}/* {backup_dir}/
            print(f"✓ Checkpoint 已備份至 Google Drive")
            
        else:
            print(f"\n❌ K={k} 訓練失敗")
            results[k] = {'status': 'failed', 'time': exp_duration}
            
    except Exception as e:
        print(f"\n❌ K={k} 發生錯誤: {str(e)}")
        results[k] = {'status': 'error', 'time': time.time() - exp_start}
    
    print("\n")

total_time = time.time() - start_time

print("=" * 60)
print("📊 訓練總結")
print("=" * 60)
print(f"總耗時: {total_time/3600:.2f} 小時")
print("\n結果:")
for k, result in results.items():
    status_icon = "✅" if result['status'] == 'success' else "❌"
    print(f"  {status_icon} K={k}: {result['status']} ({result['time']/3600:.2f}h)")

## 3. 評估結果

### 3.1 檢查 Checkpoint

In [ ]:
import os

print("📁 Checkpoint 狀態:\n")

for k in K_VALUES:
    ckpt_dir = f"checkpoints/experiments/S2_qr_K{k}"
    if os.path.exists(ckpt_dir):
        best_model = f"{ckpt_dir}/best_model.pth"
        if os.path.exists(best_model):
            size = os.path.getsize(best_model) / (1024**2)  # MB
            print(f"✓ K={k}: {best_model} ({size:.1f} MB)")
        else:
            print(f"⚠ K={k}: 目錄存在但缺少 best_model.pth")
    else:
        print(f"✗ K={k}: Checkpoint 不存在")

### 3.2 執行統一評估

In [ ]:
# 構建評估指令
checkpoint_paths = [f"checkpoints/experiments/S2_qr_K{k}/best_model.pth" for k in K_VALUES]
labels = [f"K={k}" for k in K_VALUES]

# 檢查哪些 checkpoint 存在
valid_checkpoints = []
valid_labels = []
for ckpt, label in zip(checkpoint_paths, labels):
    if os.path.exists(ckpt):
        valid_checkpoints.append(ckpt)
        valid_labels.append(label)

if len(valid_checkpoints) == 0:
    print("❌ 沒有可用的 checkpoint 進行評估")
else:
    print(f"將評估 {len(valid_checkpoints)} 個模型: {valid_labels}\n")
    
    cmd = [
        "python", "scripts/evaluate_unified.py",
        "--checkpoints"
    ] + valid_checkpoints + [
        "--labels"
    ] + valid_labels + [
        "--output", "results/S2_k_scan_comparison.png"
    ]
    
    result = subprocess.run(cmd, capture_output=False, text=True)
    
    if result.returncode == 0:
        print("\n✅ 評估完成")
        
        # 備份結果到 Google Drive
        !cp -r results /content/drive/MyDrive/pinns_results/
        print("✓ 結果已備份至 Google Drive")
    else:
        print("\n❌ 評估失敗")

### 3.3 顯示結果圖表

In [ ]:
from IPython.display import Image, display
import os

result_img = "results/S2_k_scan_comparison.png"

if os.path.exists(result_img):
    print("📊 K-Scan 比較結果:\n")
    display(Image(filename=result_img))
else:
    print(f"❌ 結果圖表不存在: {result_img}")

## 4. 下載結果

### 4.1 打包結果文件

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_name = f"s2_k_scan_results_{timestamp}"

# 創建臨時目錄
temp_dir = f"/content/{archive_name}"
os.makedirs(temp_dir, exist_ok=True)

# 複製重要文件
for k in K_VALUES:
    ckpt_dir = f"checkpoints/experiments/S2_qr_K{k}"
    if os.path.exists(ckpt_dir):
        shutil.copytree(ckpt_dir, f"{temp_dir}/S2_K{k}")

if os.path.exists("results"):
    shutil.copytree("results", f"{temp_dir}/results")

if os.path.exists("logs"):
    shutil.copytree("logs", f"{temp_dir}/logs")

# 壓縮
shutil.make_archive(f"/content/{archive_name}", 'zip', temp_dir)

print(f"✓ 結果已打包: /content/{archive_name}.zip")
print(f"  檔案大小: {os.path.getsize(f'/content/{archive_name}.zip')/(1024**2):.1f} MB")
print(f"\n下載指令: 從 Colab 檔案瀏覽器下載 {archive_name}.zip")

### 4.2 清理臨時文件（可選）

In [ ]:
# 取消註解以清理檔案（謹慎使用！）
# !rm -rf /content/pinns-sparse-flow/checkpoints/*
# !rm -rf /content/pinns-sparse-flow/results/*
# !rm -rf /content/pinns-sparse-flow/logs/*

print("⚠️ 清理已註解，如需清理請取消註解後執行")

## 5. 注意事項

### Colab 使用建議

1. **定期保存**: 每個實驗完成後自動備份至 Google Drive
2. **會話斷開**: Colab 免費版有 12 小時限制，建議:
   - 使用 Colab Pro（24 小時）
   - 分批執行（先跑 K=30,50，再跑 K=80,100）
   - 使用 `EPOCHS=5000` 快速測試
3. **GPU 檢查**: 確保使用 GPU Runtime（Runtime → Change runtime type → GPU）
4. **記憶體管理**: T4 GPU 有 16GB VRAM，通常足夠

### 實驗順序建議

**第一次執行**（快速驗證，2-4h）:
```python
K_VALUES = [30, 50]
EPOCHS = 3000
```

**第二次執行**（完整實驗，8-12h）:
```python
K_VALUES = [30, 50, 80, 100]
EPOCHS = 5000
```

**最終版本**（論文用，Colab Pro 推薦）:
```python
K_VALUES = [30, 50, 80, 100]
EPOCHS = 10000
```

### 問題排查

- **OOM (Out of Memory)**: 減少 `batch_size`
- **訓練太慢**: 檢查是否使用 GPU
- **數據缺失**: 確認已上傳 DNS 數據和 sensor 文件
- **Checkpoint 消失**: 確認 Google Drive 已掛載